# AssemblyAI STT Pipeline — Whole-Meeting AMI Dataset

This notebook is the AssemblyAI speech-to-text stage for the project.

## Input

It reads the centralized manifest created by `preprocessing.ipynb`:

```text
dataset/metadata.csv
```

The manifest points to the processed whole-meeting WAV files under `dataset/processed/`.

## Output

AssemblyAI results are intentionally **not split into train/test folders**. All meetings go directly into:

```text
outputs/assemblyai/
├── json/
├── transcripts/
└── assemblyai_summary.csv
```

The original `split` value from `dataset/metadata.csv` is retained in the JSON metadata and summary CSV so evaluation can still distinguish train and test meetings later.

### Selected meetings from preprocessing

- ES2004a
- IS1009a
- TS3003a
- EN2002a

### AssemblyAI configuration

- Speaker diarization: enabled
- Language detection: enabled
- Word-level timestamps: preserved
- Speech models: `universal-3-5-pro` with `universal-2` fallback
- Local processed WAV files are sent directly to AssemblyAI


## 1. Install required libraries

Run this once in the notebook environment. The version constraint is quoted so the shell does not interpret `>=` as a redirection/operator.

In [21]:
%pip -q install "assemblyai>=1.0.0" python-dotenv pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [22]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

from assemblyai.prerecorded.v2 import Transcriber, TranscriptionConfig

print("Imports successful.")

Imports successful.


## 3. Project configuration

The preprocessing notebook creates the central manifest at `dataset/metadata.csv`. This notebook reads that manifest instead of rebuilding the dataset mapping.

In [23]:
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "dataset"
METADATA_PATH = DATASET_DIR / "metadata.csv"

OUTPUT_DIR = BASE_DIR / "outputs" / "assemblyai"
JSON_DIR = OUTPUT_DIR / "json"
TRANSCRIPT_DIR = OUTPUT_DIR / "transcripts"
SUMMARY_PATH = OUTPUT_DIR / "assemblyai_summary.csv"

JSON_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

# Set False when you want every meeting submitted again.
SKIP_EXISTING = True

# Useful for a one-meeting test run. Set to None for all meetings.
ONE_MEETING_ID = None

# Current AssemblyAI async speech-model routing.
SPEECH_MODELS = ["universal-3-5-pro", "universal-2"]

print("Project directory :", BASE_DIR)
print("Metadata          :", METADATA_PATH)
print("JSON output       :", JSON_DIR)
print("Transcript output :", TRANSCRIPT_DIR)

Project directory : /Users/manish/Documents/KU/SEM 5/DL/Mini Project
Metadata          : /Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/metadata.csv
JSON output       : /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json
Transcript output : /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/transcripts


## 4. Load environment variables and AssemblyAI API key

Keep the API key in `.env` as:

```text
ASSEMBLYAI_API_KEY=your_key_here
```

The key is never written into the notebook outputs or saved JSON.

In [24]:
load_dotenv(BASE_DIR / ".env")
API_KEY = os.getenv("ASSEMBLYAI_API_KEY", "").strip()

if not API_KEY:
    raise RuntimeError(
        "ASSEMBLYAI_API_KEY was not found. Add it to the project .env file."
    )

print("AssemblyAI API key loaded.")

AssemblyAI API key loaded.


## 5. Load and validate the preprocessing manifest

The AssemblyAI stage uses the `processed_audio` paths from the preprocessing notebook. The train/test split is read only as metadata; it does **not** control the AssemblyAI output directories.

In [25]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {METADATA_PATH}. Run preprocessing.ipynb first."
    )

metadata_df = pd.read_csv(METADATA_PATH)

required_columns = {
    "meeting_id",
    "split",
    "duration_s",
    "processed_audio",
}
missing_columns = required_columns - set(metadata_df.columns)
if missing_columns:
    raise ValueError(f"metadata.csv is missing columns: {sorted(missing_columns)}")

if ONE_MEETING_ID is not None:
    selected_df = metadata_df[metadata_df["meeting_id"] == ONE_MEETING_ID].copy()
    if selected_df.empty:
        raise ValueError(f"Meeting {ONE_MEETING_ID!r} not found in metadata.csv")
else:
    selected_df = metadata_df.copy()

selected_df = selected_df.sort_values("meeting_id").reset_index(drop=True)

for _, row in selected_df.iterrows():
    audio_path = Path(row["processed_audio"])
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Processed audio not found for {row['meeting_id']}: {audio_path}"
        )

print(f"Meetings selected: {len(selected_df)}")
display(selected_df[["meeting_id", "split", "duration_min", "processed_audio"]])

Meetings selected: 14


,meeting_id,split,duration_min,processed_audio
0,EN2002a,test,35.711833,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
1,ES2004a,train,17.489166,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
2,IS1009a,train,13.983333,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
3,MTG_32040,train,5.986667,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
4,MTG_32063,train,6.099500,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
5,MTG_32072,train,6.090667,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
6,MTG_32074,train,6.033167,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
7,MTG_32092,train,5.974667,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
8,MTG_32179,test,5.994000,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...
9,MTG_32185,test,5.995500,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...


## 6. Create the AssemblyAI client and transcription configuration

Speaker diarization is enabled so the returned utterances include speaker labels, timings, and word-level information.

In [26]:
transcriber = Transcriber(api_key=API_KEY)

config = TranscriptionConfig(
    speech_models=SPEECH_MODELS,
    speaker_labels=True,
    language_detection=True,
    punctuate=True,
    format_text=True,
)

print("Speech models:", SPEECH_MODELS)
print("Speaker diarization: enabled")
print("Language detection : enabled")

Speech models: ['universal-3-5-pro', 'universal-2']
Speaker diarization: enabled
Language detection : enabled


## 7. Helper functions

These functions keep the saved JSON consistent and make it convenient for the later T2T notebooks.

In [27]:
def object_to_dict(obj):
    """Best-effort conversion of SDK model objects to plain Python data."""
    if obj is None:
        return None
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    if isinstance(obj, dict):
        return obj
    return dict(obj) if hasattr(obj, "__iter__") else obj


def normalize_word(word):
    return {
        "text": str(getattr(word, "text", "")),
        "offset": int(getattr(word, "start", 0)),
        "duration": max(0, int(getattr(word, "end", 0)) - int(getattr(word, "start", 0))),
        "confidence": float(getattr(word, "confidence", 0.0)),
    }


def normalize_utterance(utterance):
    words = [normalize_word(word) for word in (getattr(utterance, "words", None) or [])]
    start = int(getattr(utterance, "start", 0))
    end = int(getattr(utterance, "end", start))

    return {
        "offset": start,
        "duration": max(0, end - start),
        "speaker": getattr(utterance, "speaker", None),
        "confidence": float(getattr(utterance, "confidence", 0.0)),
        "text": str(getattr(utterance, "text", "")).strip(),
        "words": words,
    }


def build_saved_json(meeting_row, transcript, elapsed_s):
    raw_response = object_to_dict(getattr(transcript, "json_response", None))
    utterances = [normalize_utterance(u) for u in (getattr(transcript, "utterances", None) or [])]
    speakers = sorted({u["speaker"] for u in utterances if u["speaker"] is not None})

    language_code = None
    language_confidence = None
    speech_model_used = None

    if isinstance(raw_response, dict):
        language_code = raw_response.get("language_code")
        language_confidence = raw_response.get("language_confidence")
        speech_model_used = raw_response.get("speech_model_used")

    return {
        "meeting_id": str(meeting_row["meeting_id"]),
        "split": str(meeting_row["split"]),
        "engine": "AssemblyAI",
        "speech_models_requested": SPEECH_MODELS,
        "speech_model_used": speech_model_used,
        "source_audio": str(Path(meeting_row["processed_audio"]).resolve()),
        "status": str(getattr(transcript, "status", "completed")),
        "transcript_id": getattr(transcript, "id", None),
        "duration_s": float(meeting_row["duration_s"]),
        "language": language_code,
        "language_confidence": language_confidence,
        "speaker_diarization": True,
        "speakers": speakers,
        "num_speakers": len(speakers),
        "num_segments": len(utterances),
        "transcription_time_s": round(elapsed_s, 3),
        "text": str(getattr(transcript, "text", "") or "").strip(),
        "segments": utterances,
        "raw_response": raw_response,
    }


def save_result(meeting_row, transcript, elapsed_s):
    meeting_id = str(meeting_row["meeting_id"])
    saved = build_saved_json(meeting_row, transcript, elapsed_s)

    json_path = JSON_DIR / f"{meeting_id}.json"
    txt_path = TRANSCRIPT_DIR / f"{meeting_id}.txt"

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(saved, f, ensure_ascii=False, indent=2)

    with txt_path.open("w", encoding="utf-8") as f:
        for segment in saved["segments"]:
            start_s = segment["offset"] / 1000.0
            end_s = (segment["offset"] + segment["duration"]) / 1000.0
            speaker = segment["speaker"] or "?"
            f.write(f"[{start_s:.2f}s - {end_s:.2f}s] {speaker}: {segment['text']}\n")

    return json_path, txt_path, saved

## 8. Transcribe the meetings

Each selected whole-meeting WAV is sent to AssemblyAI. Existing JSON + transcript pairs are skipped when `SKIP_EXISTING=True`. A failure for one meeting is recorded and does not stop the remaining batch.

In [28]:
results = []

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="AssemblyAI transcription"):
    meeting_id = str(row["meeting_id"])
    audio_path = Path(row["processed_audio"])
    json_path = JSON_DIR / f"{meeting_id}.json"
    txt_path = TRANSCRIPT_DIR / f"{meeting_id}.txt"

    if SKIP_EXISTING and json_path.exists() and txt_path.exists():
        try:
            existing = json.loads(json_path.read_text(encoding="utf-8"))
            results.append({
                "meeting_id": meeting_id,
                "split": row["split"],
                "status": "skipped_existing",
                "transcript_id": existing.get("transcript_id"),
                "duration_s": row["duration_s"],
                "num_speakers": existing.get("num_speakers"),
                "num_segments": existing.get("num_segments"),
                "transcription_time_s": existing.get("transcription_time_s"),
                "speech_model_used": existing.get("speech_model_used"),
                "json_path": str(json_path),
                "transcript_path": str(txt_path),
                "error": None,
            })
            continue
        except Exception:
            # Re-run if an existing file is malformed or incomplete.
            pass

    start = time.perf_counter()

    try:
        transcript = transcriber.transcribe(str(audio_path), config)
        elapsed_s = time.perf_counter() - start

        if str(getattr(transcript, "status", "")).lower() == "error":
            raise RuntimeError(getattr(transcript, "error", "Unknown AssemblyAI error"))

        json_path, txt_path, saved = save_result(row, transcript, elapsed_s)

        results.append({
            "meeting_id": meeting_id,
            "split": row["split"],
            "status": "completed",
            "transcript_id": saved.get("transcript_id"),
            "duration_s": row["duration_s"],
            "num_speakers": saved.get("num_speakers"),
            "num_segments": saved.get("num_segments"),
            "transcription_time_s": saved.get("transcription_time_s"),
            "speech_model_used": saved.get("speech_model_used"),
            "json_path": str(json_path),
            "transcript_path": str(txt_path),
            "error": None,
        })

        print(f"\nCompleted: {meeting_id} | {elapsed_s / 60:.2f} min")

    except Exception as exc:
        elapsed_s = time.perf_counter() - start
        error_text = f"{type(exc).__name__}: {exc}"

        results.append({
            "meeting_id": meeting_id,
            "split": row["split"],
            "status": "error",
            "transcript_id": None,
            "duration_s": row["duration_s"],
            "num_speakers": None,
            "num_segments": None,
            "transcription_time_s": round(elapsed_s, 3),
            "speech_model_used": None,
            "json_path": str(json_path),
            "transcript_path": str(txt_path),
            "error": error_text,
        })

        print(f"\nERROR: {meeting_id} -> {error_text}")

AssemblyAI transcription:   0%|          | 0/14 [00:00<?, ?it/s]


Completed: MTG_32040 | 8.05 min

Completed: MTG_32063 | 4.89 min

Completed: MTG_32072 | 2.44 min

Completed: MTG_32074 | 2.26 min

Completed: MTG_32092 | 0.93 min

Completed: MTG_32179 | 0.82 min

Completed: MTG_32185 | 1.16 min

Completed: MTG_32256 | 0.67 min

Completed: MTG_32257 | 0.75 min

Completed: MTG_32322 | 0.70 min


## 9. Save the batch summary

The summary is flat and keeps the original train/test membership only as a metadata column.

In [29]:
summary_df = pd.DataFrame(results).sort_values("meeting_id").reset_index(drop=True)
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")

display(summary_df)
print("\nSaved:", SUMMARY_PATH)

,meeting_id,split,status,transcript_id,duration_s,num_speakers,num_segments,transcription_time_s,speech_model_used,json_path,transcript_path,error
0,EN2002a,test,skipped_existing,94758bfe-136a-48e2-9c85-4afbe135f28d,2142.709961,4,411,139.498,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
1,ES2004a,train,skipped_existing,94ddf46d-50d8-4d2d-a0c3-99cadddcce89,1049.349976,4,97,121.704,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
2,IS1009a,train,skipped_existing,cf6db210-c339-4910-ae09-2d1b8c7cbcd9,839.000000,5,103,55.194,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
3,MTG_32040,train,completed,bdd694a9-a85b-4394-a0bc-528e56f3d080,359.200012,5,81,483.225,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
4,MTG_32063,train,completed,03ae954c-7294-4d60-9910-305802eef45e,365.970001,3,72,293.483,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
5,MTG_32072,train,completed,17cfdbb9-f8ef-4550-b35b-40303d8498ea,365.440002,3,60,146.428,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
6,MTG_32074,train,completed,a211cc9a-6e26-4bf8-bae7-b708317dbcf1,361.989990,3,68,135.764,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
7,MTG_32092,train,completed,a03f95f7-ce8f-47c0-a6ab-bb3fd231cfbb,358.480011,5,45,56.093,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
8,MTG_32179,test,completed,d6eb58a9-6254-4257-bf4a-ed8bbfe9d2ab,359.640015,6,77,48.992,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None
9,MTG_32185,test,completed,5920dc9d-537e-4ba6-9cae-58d5e08a119d,359.730011,5,17,69.779,universal-3-5-pro,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,/Users/manish/Documents/KU/SEM 5/DL/Mini Proje...,None



Saved: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/assemblyai_summary.csv


## 10. Validate generated files

A successful meeting should have both a JSON file and a readable transcript file.

In [30]:
validation_rows = []

for _, row in summary_df.iterrows():
    meeting_id = row["meeting_id"]
    json_path = JSON_DIR / f"{meeting_id}.json"
    txt_path = TRANSCRIPT_DIR / f"{meeting_id}.txt"

    json_ok = json_path.exists()
    txt_ok = txt_path.exists()
    json_valid = False
    text_nonempty = False
    segment_count = None

    if json_ok:
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
            json_valid = isinstance(data, dict)
            segment_count = len(data.get("segments", []))
            text_nonempty = bool(str(data.get("text", "")).strip())
        except json.JSONDecodeError:
            json_valid = False

    validation_rows.append({
        "meeting_id": meeting_id,
        "status": row["status"],
        "json_exists": json_ok,
        "json_valid": json_valid,
        "transcript_exists": txt_ok,
        "text_nonempty": text_nonempty,
        "segments": segment_count,
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

completed = validation_df[validation_df["status"].isin(["completed", "skipped_existing"])].copy()
if not completed.empty:
    if not completed["json_exists"].all() or not completed["json_valid"].all():
        raise ValueError("At least one successful meeting has an invalid/missing JSON output.")
    if not completed["transcript_exists"].all() or not completed["text_nonempty"].all():
        raise ValueError("At least one successful meeting has a missing/empty transcript.")

print("Output validation completed.")

,meeting_id,status,json_exists,json_valid,transcript_exists,text_nonempty,segments
0,EN2002a,skipped_existing,True,True,True,True,411
1,ES2004a,skipped_existing,True,True,True,True,97
2,IS1009a,skipped_existing,True,True,True,True,103
3,MTG_32040,completed,True,True,True,True,81
4,MTG_32063,completed,True,True,True,True,72
5,MTG_32072,completed,True,True,True,True,60
6,MTG_32074,completed,True,True,True,True,68
7,MTG_32092,completed,True,True,True,True,45
8,MTG_32179,completed,True,True,True,True,77
9,MTG_32185,completed,True,True,True,True,17


Output validation completed.


## 11. Inspect one AssemblyAI result

This gives a quick view of the speaker-aware transcript that will later feed the T2T stage.

In [31]:
successful_ids = [
    r["meeting_id"]
    for r in results
    if r["status"] in {"completed", "skipped_existing"}
]

if successful_ids:
    sample_id = successful_ids[0]
    sample_json_path = JSON_DIR / f"{sample_id}.json"
    sample_data = json.loads(sample_json_path.read_text(encoding="utf-8"))

    print("Meeting:", sample_data["meeting_id"])
    print("Split:", sample_data["split"])
    print("Duration:", round(sample_data["duration_s"] / 60, 2), "minutes")
    print("Speakers:", ", ".join(sample_data["speakers"]))
    print("Segments:", sample_data["num_segments"])
    print("Model used:", sample_data.get("speech_model_used"))
    print("\nFirst 10 segments:\n")

    for segment in sample_data["segments"][:10]:
        start_s = segment["offset"] / 1000.0
        end_s = (segment["offset"] + segment["duration"]) / 1000.0
        print(f"[{start_s:7.2f}s - {end_s:7.2f}s] {segment['speaker']}: {segment['text']}")
else:
    print("No successful AssemblyAI results are available to inspect.")

Meeting: EN2002a
Split: test
Duration: 35.71 minutes
Speakers: A, B, C, D
Segments: 411
Model used: universal-3-5-pro

First 10 segments:

[   0.53s -    3.55s] A: I wonder how much of the meetings is talking about the stuff at the meetings.
[   3.66s -   12.01s] B: Yeah, exactly. Yeah, yeah. Look at all this stuff. Okay, welcome to start. Okay.
[  12.10s -   15.32s] C: Right. So should we need to talk about it?
[  15.50s -   18.96s] B: Should we just go around and everyone says what they what they've been doing?
[  19.14s -   22.06s] D: Has anybody done anything? No one.
[  22.11s -   24.97s] B: Well, I'm sorry.
[  25.83s -   25.96s] C: Yeah.
[  26.29s -   27.95s] A: Okay. Sounds like you've done something.
[  29.93s -   30.33s] C: Okay.
[  30.51s -   36.07s] B: Well, I've got a browser now, which whoops already dropped.


## 12. Final output structure

This notebook does **not** create `train/` or `test/` directories under AssemblyAI outputs.

Expected structure:

```text
outputs/assemblyai/
├── json/
│   ├── ES2004a.json
│   ├── IS1009a.json
│   ├── TS3003a.json
│   └── EN2002a.json
├── transcripts/
│   ├── ES2004a.txt
│   ├── IS1009a.txt
│   ├── TS3003a.txt
│   └── EN2002a.txt
└── assemblyai_summary.csv
```

The next stage can read `outputs/assemblyai/transcripts/` directly for BART and FLAN-T5, while `outputs/assemblyai/json/` preserves the speaker/timestamp information.

In [32]:
print("=" * 72)
print("ASSEMBLYAI STT COMPLETE")
print("=" * 72)
print(f"Meetings processed : {len(summary_df)}")
print(f"Completed          : {(summary_df['status'] == 'completed').sum()}")
print(f"Skipped existing   : {(summary_df['status'] == 'skipped_existing').sum()}")
print(f"Errors             : {(summary_df['status'] == 'error').sum()}")
print(f"JSON directory     : {JSON_DIR}")
print(f"Transcript directory: {TRANSCRIPT_DIR}")
print(f"Summary CSV        : {SUMMARY_PATH}")

ASSEMBLYAI STT COMPLETE
Meetings processed : 14
Completed          : 10
Skipped existing   : 4
Errors             : 0
JSON directory     : /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json
Transcript directory: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/transcripts
Summary CSV        : /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/assemblyai_summary.csv


## Notes and references

The implementation follows AssemblyAI's current Python SDK pattern for pre-recorded audio: local files can be passed to the transcriber, speaker diarization is enabled with `speaker_labels=True`, language detection can be enabled, and the completed transcript exposes speaker-labelled utterances and word-level timestamps. The current SDK examples use `assemblyai.prerecorded.v2` and `assemblyai>=1.0.0`.

References:
- AssemblyAI Python SDK / pre-recorded v2 examples: https://www.assemblyai.com/blog/apply-llms-multi-speaker-audio
- AssemblyAI SDK JSON export guidance: https://www.assemblyai.com/docs/pre-recorded-audio/guides/do-more-with-sdk
- AssemblyAI speaker diarization: https://www.assemblyai.com/docs/pre-recorded-audio/label-speakers
